# Encoder Comparison — WavLM vs Wav2Vec2 vs Whisper-Medium

Same pipeline shape as `wavlm_comparison.ipynb`:
- Each folder has `{folder}GT.csv` with `filename,label` next to the notebook.
- Per-folder cache CSVs live in the notebook directory (`NB_DIR`):
  - `{folder}_wavlm_whole.csv`     — 768-dim mean-pooled WavLM-base-plus
  - `{folder}_wav2vec2_whole.csv`  — 768-dim mean-pooled Wav2Vec2-base
  - `{folder}_whisper_whole.csv`   — 1024-dim mean-pooled Whisper-medium encoder (30s chunks)
- WavLM cell also reuses `{folder}_features_wavlm.csv` or `{folder}_wavlm.csv` if already present.

**Candidate isolation:** filenames are `{candidate_id}_25.wav` / `_26.wav` / `_27.wav` → candidate_id = `filename.rsplit('_', 1)[0]`. No candidate appears in both train and test.

**Order on CPU (slowest last):** WavLM → Wav2Vec2 → Whisper-medium → overall table.

In [ ]:
# ============================================================
# CONFIG — edit this cell only
# ============================================================
from pathlib import Path

TRAIN_FOLDERS = ["audios2", "audios4"]
TEST_FOLDER   = "audios5"   # empty "" -> 20% candidate-grouped holdout

WAVLM_MODEL    = "microsoft/wavlm-base-plus"
WAV2VEC2_MODEL = "facebook/wav2vec2-base"
WHISPER_MODEL  = "openai/whisper-medium"

WHISPER_CHUNK_SEC = 30
MAX_DURATION_SEC  = 120
SR                = 16000

TEST_RATIO  = 0.20
RANDOM_SEED = 42
AUDIO_EXTS  = {".wav", ".mp3", ".m4a", ".flac", ".ogg", ".wma", ".aac", ".webm", ".mp4"}

SAVE_DIR = "checkpoints_encoder_cmp"
NB_DIR   = Path(".").resolve()
SAVE_DIR = NB_DIR / SAVE_DIR
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Train: {TRAIN_FOLDERS}  |  Test: {TEST_FOLDER or f'{int(TEST_RATIO*100)}% candidate-grouped holdout'}")
print(f"NB_DIR: {NB_DIR}")
print(f"Save:   {SAVE_DIR}")

In [ ]:
import os, json, gc, warnings, shutil as _shutil
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import torch
import joblib
import xgboost as xgb
from tqdm import tqdm
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             accuracy_score, confusion_matrix, classification_report)

warnings.filterwarnings('ignore')
np.random.seed(RANDOM_SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

LABEL_MAP = {
    "cheating":1,"read":1,"reading":1,"scripted":1,"yes":1,"1":1,1:1,
    "not cheating":0,"not_cheating":0,"spontaneous":0,"no":0,"0":0,0:0,"genuine":0,
}

In [ ]:
def candidate_id(filename):
    stem = Path(filename).stem
    parts = stem.rsplit('_', 1)
    return parts[0] if len(parts) == 2 else stem

def load_audio_16k(path, max_sec=MAX_DURATION_SEC):
    try:
        y, sr = sf.read(str(path), always_2d=False)
        if y.ndim > 1: y = y.mean(axis=1)
        if sr != SR:
            y = librosa.resample(y.astype(np.float32), orig_sr=sr, target_sr=SR)
        y = y.astype(np.float32)
        if max_sec and len(y) > max_sec * SR:
            y = y[:int(max_sec * SR)]
        return y
    except Exception as e:
        print(f'  WARN {Path(path).name}: {e}')
        return None

def scan_folder(name):
    audio_dir = NB_DIR / name
    if not audio_dir.exists():
        print(f'  SKIP {name}: folder not found'); return None
    files = sorted(f for f in audio_dir.rglob('*')
                   if f.suffix.lower() in AUDIO_EXTS and f.is_file())
    if not files:
        print(f'  SKIP {name}: no audio files'); return None
    gt_path = NB_DIR / f'{name}GT.csv'
    if not gt_path.exists():
        print(f'  SKIP {name}: no GT file'); return None
    return {
        'name':     name,
        'files':    files,
        'gt':       gt_path,
        'wavlm':    NB_DIR / f'{name}_wavlm_whole.csv',
        'wav2vec2': NB_DIR / f'{name}_wav2vec2_whole.csv',
        'whisper':  NB_DIR / f'{name}_whisper_whole.csv',
    }

def load_gt(gt_path):
    gt = pd.read_csv(gt_path)
    fn_col  = next((c for c in gt.columns if c.lower() in ('filename','file','name')), gt.columns[0])
    lbl_col = next((c for c in gt.columns if c.lower() in ('label','class','cheating','gt','label_int','ground_truth')), gt.columns[-1])
    gt = gt.rename(columns={fn_col: 'filename', lbl_col: 'label_raw'})
    gt['label_int'] = gt['label_raw'].map(lambda x: LABEL_MAP.get(x, LABEL_MAP.get(str(x).lower().strip(), -1)))
    gt = gt[gt['label_int'].isin([0,1])][['filename','label_int']].copy()
    gt['label_int'] = gt['label_int'].astype(int)
    return gt

all_names = list(dict.fromkeys(TRAIN_FOLDERS + ([TEST_FOLDER] if TEST_FOLDER else [])))
folders   = [m for m in (scan_folder(n) for n in all_names) if m]

print(f"\n{'folder':<12s} {'files':>6s}  {'wavlm':>8s}  {'w2v2':>8s}  {'whisper':>8s}")
for m in folders:
    def tag(p): return 'cached' if p.exists() else 'need'
    print(f"{m['name']:<12s} {len(m['files']):>6d}  {tag(m['wavlm']):>8s}  "
          f"{tag(m['wav2vec2']):>8s}  {tag(m['whisper']):>8s}")

## 1. Build Master Index + Candidate-Isolated Split
One split reused for all three encoders so results are directly comparable.

In [ ]:
rows = []
for m in folders:
    gt = load_gt(m['gt'])
    gt['folder'] = m['name']
    gt['candidate_id'] = gt['filename'].map(candidate_id)
    rows.append(gt)
master = pd.concat(rows, ignore_index=True)
print(f"Master index: {len(master)} rows, {master['candidate_id'].nunique()} unique candidates")

if TEST_FOLDER:
    test_mask  = master['folder'] == TEST_FOLDER
    test_cands = set(master.loc[test_mask, 'candidate_id'])
    train_mask = (master['folder'].isin([n for n in TRAIN_FOLDERS if n != TEST_FOLDER])
                  & ~master['candidate_id'].isin(test_cands))
    dropped = ((master['folder'].isin([n for n in TRAIN_FOLDERS if n != TEST_FOLDER]))
               & master['candidate_id'].isin(test_cands)).sum()
    print(f"Explicit test folder: {TEST_FOLDER}  |  dropped {int(dropped)} leak rows from train")
else:
    gss = GroupShuffleSplit(n_splits=1, test_size=TEST_RATIO, random_state=RANDOM_SEED)
    tr_idx, te_idx = next(gss.split(master, groups=master['candidate_id']))
    train_mask = np.zeros(len(master), dtype=bool); train_mask[tr_idx] = True
    test_mask  = np.zeros(len(master), dtype=bool); test_mask[te_idx]  = True
    print(f"GroupShuffleSplit on candidate_id (test_size={TEST_RATIO})")

train_files = set(master.loc[train_mask, 'filename'])
test_files  = set(master.loc[test_mask,  'filename'])
assert not (train_files & test_files), 'filename overlap between train/test'

train_cands = set(master.loc[master['filename'].isin(train_files), 'candidate_id'])
test_cands_final = set(master.loc[master['filename'].isin(test_files), 'candidate_id'])
assert not (train_cands & test_cands_final), 'candidate overlap -- isolation broken'

y_tr_master = master[master['filename'].isin(train_files)][['filename','label_int']]
y_te_master = master[master['filename'].isin(test_files)][['filename','label_int']]
print(f"Train: {len(train_files)} files  ({int((y_tr_master.label_int==1).sum())} cheating / "
      f"{int((y_tr_master.label_int==0).sum())} honest)   {len(train_cands)} candidates")
print(f"Test:  {len(test_files)} files  ({int((y_te_master.label_int==1).sum())} cheating / "
      f"{int((y_te_master.label_int==0).sum())} honest)   {len(test_cands_final)} candidates")

In [ ]:
PREC_TARGETS = [0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]

# --- audios4-CV configuration (matches fusion_text_wavlm.ipynb) ---
from sklearn.model_selection import StratifiedKFold
CV_TARGET_BATCH = 'audios4'
CV_FOLDS        = 5
DEPLOY_POS_RATE = 0.17
SPW_DEPLOY      = (1.0 - DEPLOY_POS_RATE) / DEPLOY_POS_RATE
print(f'audios4-CV: target={CV_TARGET_BATCH}  folds={CV_FOLDS}  SPW_DEPLOY={SPW_DEPLOY:.4f}')

def rec_at_prec_targets(proba, y, targets=PREC_TARGETS, min_tp=3):
    y = np.asarray(y)
    rows = []
    for t in np.arange(0.05, 0.991, 0.005):
        pred = (proba >= t).astype(int)
        tp = int(((pred == 1) & (y == 1)).sum())
        if tp < min_tp: continue
        rows.append((float(t),
                     float(precision_score(y, pred, zero_division=0)),
                     float(recall_score(y, pred, zero_division=0))))
    out = {}
    for tp_target in targets:
        cands = [(thr, p, r) for thr, p, r in rows if p >= tp_target]
        if cands:
            best = max(cands, key=lambda x: x[2])
            out[f'rec@P{int(tp_target*100)}'] = round(best[2], 4)
            out[f'thr@P{int(tp_target*100)}'] = round(best[0], 3)
        else:
            out[f'rec@P{int(tp_target*100)}'] = 0.0
            out[f'thr@P{int(tp_target*100)}'] = None
    return out

def threshold_sweep(proba, y):
    best_thr, best_f1 = 0.5, 0.0
    for thr in np.arange(0.20, 0.81, 0.02):
        f = f1_score(y, (proba >= thr).astype(int), zero_division=0)
        if f > best_f1: best_f1, best_thr = f, thr
    return round(best_thr, 2), round(best_f1, 4)

def coarse_sweep(proba, y):
    rows = []
    for thr in np.arange(0.20, 0.81, 0.05):
        pred = (proba >= thr).astype(int)
        cm = confusion_matrix(y, pred, labels=[0,1])
        rows.append(dict(thr=round(float(thr),2),
            prec=round(precision_score(y, pred, zero_division=0),4),
            rec=round(recall_score(y, pred, zero_division=0),4),
            f1=round(f1_score(y, pred, zero_division=0),4),
            tp=int(cm[1,1]), fp=int(cm[0,1]), fn=int(cm[1,0]), tn=int(cm[0,0])))
    return pd.DataFrame(rows)

# ---------- audios4-CV helpers ----------
def _best_f1(proba, y, grid=np.arange(0.20, 0.81, 0.02)):
    bt, bf = 0.5, 0.0
    for t in grid:
        f = f1_score(y, (proba >= t).astype(int), zero_division=0)
        if f > bf: bf, bt = f, t
    return float(bt), float(bf)

def audios4_cv_oof(build_clf, X_always, y_always, X_cv, y_cv,
                   folds=CV_FOLDS, seed=RANDOM_SEED):
    oof = np.full(len(X_cv), np.nan)
    fa  = np.full(len(X_cv), -1, dtype=int)
    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=seed)
    for fi, (tr_idx, va_idx) in enumerate(skf.split(X_cv, y_cv)):
        if len(X_always):
            Xtr = np.vstack([X_always, X_cv[tr_idx]])
            ytr = np.concatenate([y_always, y_cv[tr_idx]])
        else:
            Xtr, ytr = X_cv[tr_idx], y_cv[tr_idx]
        sc  = StandardScaler().fit(Xtr)
        clf = build_clf()
        clf.fit(sc.transform(Xtr), ytr)
        oof[va_idx] = clf.predict_proba(sc.transform(X_cv[va_idx]))[:, 1]
        fa[va_idx]  = fi
    return oof, fa

def cv_metrics_from_oof(oof, y, fold_assign):
    thr_g, f1_g = _best_f1(oof, y)
    pred = (oof >= thr_g).astype(int)
    global_m = dict(
        cv_thr=round(thr_g, 3),
        cv_f1=round(f1_g, 4),
        cv_prec=round(precision_score(y, pred, zero_division=0), 4),
        cv_rec=round(recall_score(y, pred, zero_division=0), 4),
    )
    per_fold = []
    for fi in sorted(set(fold_assign.tolist())):
        mk = fold_assign == fi
        y_f, p_f = y[mk], oof[mk]
        pred_f = (p_f >= thr_g).astype(int)
        own_thr, own_f1 = _best_f1(p_f, y_f)
        per_fold.append(dict(
            fold=int(fi), n=int(mk.sum()), n_cheat=int((y_f==1).sum()),
            f1_at_global=round(f1_score(y_f, pred_f, zero_division=0), 4),
            prec_at_global=round(precision_score(y_f, pred_f, zero_division=0), 4),
            rec_at_global=round(recall_score(y_f, pred_f, zero_division=0), 4),
            own_best_thr=round(own_thr, 3),
            own_best_f1=round(own_f1, 4),
        ))
    pf = pd.DataFrame(per_fold)
    agg = dict(
        fold_f1_mean=round(float(pf['f1_at_global'].mean()), 4),
        fold_f1_std =round(float(pf['f1_at_global'].std()),  4),
        fold_thr_mean=round(float(pf['own_best_thr'].mean()), 3),
        fold_thr_std =round(float(pf['own_best_thr'].std()),  3),
    )
    return global_m, agg, pf

def _test_metrics_at(proba_te, y_te_arr, thr):
    pred = (proba_te >= thr).astype(int)
    return dict(
        test_f1_at_cv  =round(f1_score(y_te_arr, pred, zero_division=0), 4),
        test_prec_at_cv=round(precision_score(y_te_arr, pred, zero_division=0), 4),
        test_rec_at_cv =round(recall_score(y_te_arr, pred, zero_division=0), 4),
    )

def cv_and_gap_bundle(build_clf, X_always, y_always, X_cv, y_cv,
                      X_te_arr, y_te_arr, config_tag):
    """Run audios4-CV, fit on full (audios2+audios4), score test at frozen cv_thr."""
    oof, fa = audios4_cv_oof(build_clf, X_always, y_always, X_cv, y_cv)
    g, a, pf = cv_metrics_from_oof(oof, y_cv, fa)
    if len(X_always):
        X_full = np.vstack([X_always, X_cv]); y_full = np.concatenate([y_always, y_cv])
    else:
        X_full, y_full = X_cv, y_cv
    sc  = StandardScaler().fit(X_full)
    clf = build_clf()
    clf.fit(sc.transform(X_full), y_full)
    proba_te_cv = clf.predict_proba(sc.transform(X_te_arr))[:, 1]
    t = _test_metrics_at(proba_te_cv, y_te_arr, g['cv_thr'])
    extra = {**g, **a, **t, 'gap_f1': round(g['cv_f1'] - t['test_f1_at_cv'], 4)}
    pf.insert(0, 'config', config_tag)
    return extra, pf

# ---------- main training fn (now audios4-CV-aware) ----------
def train_xgb_rf(X_tr, y_tr, X_te, y_te, tag, cv_data=None):
    sc = StandardScaler().fit(X_tr)
    Xtr, Xte = sc.transform(X_tr), sc.transform(X_te)
    spw = float((y_tr==0).sum()) / max(float((y_tr==1).sum()), 1.0)
    colsample = 0.3 if X_tr.shape[1] > 500 else 0.8

    # CV factories use SPW_DEPLOY so CV metrics are comparable across notebooks.
    cv_factories = {
        'xgb': lambda: xgb.XGBClassifier(
            n_estimators=400, max_depth=5, learning_rate=0.04,
            subsample=0.8, colsample_bytree=colsample, min_child_weight=3,
            scale_pos_weight=float(SPW_DEPLOY), eval_metric='logloss',
            random_state=RANDOM_SEED, device='cpu'),
        'rf': lambda: RandomForestClassifier(
            n_estimators=500, max_depth=None, min_samples_leaf=2,
            class_weight={0:1.0, 1:float(SPW_DEPLOY)},
            n_jobs=-1, random_state=RANDOM_SEED),
    }

    results = {}
    for mdl_name, clf in [
        ('xgb', xgb.XGBClassifier(
            n_estimators=400, max_depth=5, learning_rate=0.04,
            subsample=0.8, colsample_bytree=colsample, min_child_weight=3,
            scale_pos_weight=spw, eval_metric='logloss',
            early_stopping_rounds=30, random_state=RANDOM_SEED, device='cpu')),
        ('rf', RandomForestClassifier(
            n_estimators=500, max_depth=None, min_samples_leaf=2,
            class_weight='balanced', n_jobs=-1, random_state=RANDOM_SEED)),
    ]:
        if mdl_name == 'xgb':
            clf.fit(Xtr, y_tr, eval_set=[(Xte, y_te)], verbose=False)
        else:
            clf.fit(Xtr, y_tr)
        proba = clf.predict_proba(Xte)[:, 1]
        thr, f1 = threshold_sweep(proba, y_te)
        pred = (proba >= thr).astype(int)
        cm = confusion_matrix(y_te, pred, labels=[0,1])
        rec_p = rec_at_prec_targets(proba, y_te)
        r = dict(
            tag=f'{tag}_{mdl_name}', n_feat=X_tr.shape[1], thr=thr, f1=f1,
            precision=round(precision_score(y_te, pred, zero_division=0),4),
            recall=round(recall_score(y_te, pred, zero_division=0),4),
            accuracy=round(accuracy_score(y_te, pred),4),
            tp=int(cm[1,1]), fp=int(cm[0,1]), fn=int(cm[1,0]), tn=int(cm[0,0]),
            proba=proba, model=clf, scaler=sc,
            **rec_p,
        )
        if cv_data is not None:
            X_a, y_a, X_c, y_c = cv_data
            extra, pf = cv_and_gap_bundle(
                cv_factories[mdl_name], X_a, y_a, X_c, y_c,
                Xte, y_te, config_tag=r['tag'])
            for k, v in extra.items(): r[k] = v
            r['fold_df'] = pf
        print(f"  {r['tag']:<18s}  F1={r['f1']:.4f}  P={r['precision']:.4f}  R={r['recall']:.4f}  thr={r['thr']}  n_feat={r['n_feat']}")
        if 'cv_f1' in r:
            print(f"     audios4-CV  F1={r['cv_f1']:.4f}  P={r['cv_prec']:.4f}  R={r['cv_rec']:.4f}  cv_thr={r['cv_thr']:.3f}"
                  f"  | test@cv_thr F1={r['test_f1_at_cv']:.4f}  gap_f1={r['gap_f1']:+.4f}"
                  f"  | fold_f1={r['fold_f1_mean']:.4f}±{r['fold_f1_std']:.4f}")
        print(f"     rec @P60={r['rec@P60']:.3f}  @P65={r['rec@P65']:.3f}  @P70={r['rec@P70']:.3f}  "
              f"@P75={r['rec@P75']:.3f}  @P80={r['rec@P80']:.3f}  @P85={r['rec@P85']:.3f}  @P90={r['rec@P90']:.3f}")
        results[mdl_name] = r
    return results

def build_xy_from_cache(cache_col_prefix, folders, feat_paths_attr):
    dfs = []
    for m in folders:
        p = m[feat_paths_attr]
        if not p.exists(): continue
        dfs.append(pd.read_csv(p))
    feat_df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    feat_cols = [c for c in feat_df.columns if c.startswith(cache_col_prefix)]
    if not feat_cols:
        raise RuntimeError(f'No feature columns found with prefix {cache_col_prefix!r}')
    tr_meta = master[master['filename'].isin(train_files)][['filename','folder','label_int']]
    te_meta = master[master['filename'].isin(test_files)][['filename','folder','label_int']]
    tr = feat_df.merge(tr_meta, on='filename', how='inner')
    te = feat_df.merge(te_meta, on='filename', how='inner')
    X_tr = tr[feat_cols].fillna(0).values
    y_tr = tr['label_int'].values
    X_te = te[feat_cols].fillna(0).values
    y_te = te['label_int'].values
    return X_tr, y_tr, X_te, y_te, tr, te, feat_cols

def split_always_cv(tr_df, feat_cols):
    """Split train rows into (always-in-train) and (audios4-CV) matrices by folder."""
    mk_cv = (tr_df['folder'] == CV_TARGET_BATCH).values
    X = tr_df[feat_cols].fillna(0).values
    y = tr_df['label_int'].values
    return X[~mk_cv], y[~mk_cv], X[mk_cv], y[mk_cv]

def save_results(results_dict, te_df, y_te):
    base_keep = ['tag','n_feat','thr','f1','precision','recall','accuracy','tp','fp','fn','tn',
                 'cv_thr','cv_f1','cv_prec','cv_rec',
                 'test_f1_at_cv','test_prec_at_cv','test_rec_at_cv','gap_f1',
                 'fold_f1_mean','fold_f1_std','fold_thr_mean','fold_thr_std']
    rp_keep   = [f'rec@P{int(p*100)}' for p in PREC_TARGETS]
    keep_cols = base_keep + rp_keep
    for k, r in results_dict.items():
        all_results.append({c: r.get(c) for c in keep_cols})
        coarse_sweep(r['proba'], y_te).to_csv(SAVE_DIR / f"sweep_{r['tag']}.csv", index=False)
        pd.DataFrame({'filename': te_df['filename'].values, 'label_int': y_te, 'proba': r['proba']}).to_csv(
            SAVE_DIR / f"pred_{r['tag']}.csv", index=False)
        joblib.dump(r['scaler'], SAVE_DIR / f"scaler_{r['tag']}.pkl")
        if hasattr(r['model'], 'save_model'):
            r['model'].save_model(str(SAVE_DIR / f"model_{r['tag']}.json"))
        else:
            joblib.dump(r['model'], SAVE_DIR / f"model_{r['tag']}.pkl")
        if isinstance(r.get('fold_df'), pd.DataFrame) and not r['fold_df'].empty:
            r['fold_df'].to_csv(SAVE_DIR / f"fold_{r['tag']}.csv", index=False)
            all_folds.append(r['fold_df'])

all_results = []
all_folds   = []
print('Helpers ready (audios4-CV + gap_f1 enabled).')

## 2. WavLM — Extract + Evaluate

In [ ]:
need_wavlm = []
for m in folders:
    if m['wavlm'].exists():
        print(f"  {m['name']}: {m['wavlm'].name} cached."); continue
    for alt_name in (f"{m['name']}_features_wavlm.csv", f"{m['name']}_wavlm.csv"):
        alt = NB_DIR / alt_name
        if alt.exists():
            df_alt = pd.read_csv(alt)
            if any(c.startswith('wavlm_') for c in df_alt.columns):
                _shutil.copy(str(alt), str(m['wavlm']))
                print(f"  {m['name']}: reused {alt_name} -> {m['wavlm'].name}")
                break
    if not m['wavlm'].exists():
        need_wavlm.append(m)

if need_wavlm:
    from transformers import AutoFeatureExtractor, WavLMModel
    print(f"\nLoading {WAVLM_MODEL} ...")
    fe = AutoFeatureExtractor.from_pretrained(WAVLM_MODEL)
    mdl = WavLMModel.from_pretrained(WAVLM_MODEL).eval().to(DEVICE)
    D = mdl.config.hidden_size
    print(f"WavLM hidden dim: {D}")

    @torch.no_grad()
    def wavlm_embed(y):
        inp = fe(y, sampling_rate=SR, return_tensors='pt', padding=False)
        out = mdl(inp.input_values.to(DEVICE))
        return out.last_hidden_state.mean(dim=1).squeeze(0).cpu().numpy()

    for m in need_wavlm:
        print(f"\nExtracting WavLM: {m['name']} ({len(m['files'])} files)...")
        rows = []
        for fp in tqdm(m['files'], desc=m['name']):
            y = load_audio_16k(fp)
            if y is None: continue
            e = wavlm_embed(y)
            row = {'filename': fp.name}
            for i, v in enumerate(e): row[f'wavlm_{i}'] = round(float(v), 6)
            rows.append(row)
        pd.DataFrame(rows).to_csv(m['wavlm'], index=False)
        print(f"  Saved: {m['wavlm'].name} ({len(rows)} rows, {D} dims)")

    del mdl, fe
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    gc.collect()
else:
    print('All WavLM caches present.')

In [ ]:
X_tr, y_tr, X_te, y_te, _, te_df, _ = build_xy_from_cache('wavlm_', folders, 'wavlm')
print(f"WavLM:  train={X_tr.shape}  test={X_te.shape}")
res_wavlm = train_xgb_rf(X_tr, y_tr, X_te, y_te, tag='wavlm')
save_results(res_wavlm, te_df, y_te)

## 3. Wav2Vec2-base — Extract + Evaluate

In [ ]:
need_w2v = [m for m in folders if not m['wav2vec2'].exists()]
if need_w2v:
    from transformers import AutoFeatureExtractor, Wav2Vec2Model
    print(f"Loading {WAV2VEC2_MODEL} ...")
    fe = AutoFeatureExtractor.from_pretrained(WAV2VEC2_MODEL)
    mdl = Wav2Vec2Model.from_pretrained(WAV2VEC2_MODEL).eval().to(DEVICE)
    D = mdl.config.hidden_size
    print(f"Wav2Vec2 hidden dim: {D}")

    @torch.no_grad()
    def w2v_embed(y):
        inp = fe(y, sampling_rate=SR, return_tensors='pt', padding=False)
        out = mdl(inp.input_values.to(DEVICE))
        return out.last_hidden_state.mean(dim=1).squeeze(0).cpu().numpy()

    for m in need_w2v:
        print(f"\nExtracting Wav2Vec2: {m['name']} ({len(m['files'])} files)...")
        rows = []
        for fp in tqdm(m['files'], desc=m['name']):
            y = load_audio_16k(fp)
            if y is None: continue
            e = w2v_embed(y)
            row = {'filename': fp.name}
            for i, v in enumerate(e): row[f'wav2vec2_{i}'] = round(float(v), 6)
            rows.append(row)
        pd.DataFrame(rows).to_csv(m['wav2vec2'], index=False)
        print(f"  Saved: {m['wav2vec2'].name} ({len(rows)} rows, {D} dims)")

    del mdl, fe
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    gc.collect()
else:
    print('All Wav2Vec2 caches present.')

In [ ]:
X_tr, y_tr, X_te, y_te, _, te_df, _ = build_xy_from_cache('wav2vec2_', folders, 'wav2vec2')
print(f"Wav2Vec2:  train={X_tr.shape}  test={X_te.shape}")
res_w2v = train_xgb_rf(X_tr, y_tr, X_te, y_te, tag='wav2vec2')
save_results(res_w2v, te_df, y_te)

## 4. Whisper-Medium Encoder — Extract + Evaluate
Heaviest model; runs last. Encoder only, 30s chunks, mean-pool over chunks.

In [ ]:
need_wh = [m for m in folders if not m['whisper'].exists()]
if need_wh:
    from transformers import WhisperProcessor, WhisperModel
    print(f"Loading {WHISPER_MODEL} ...")
    proc = WhisperProcessor.from_pretrained(WHISPER_MODEL)
    mdl  = WhisperModel.from_pretrained(WHISPER_MODEL).eval().to(DEVICE)
    D = mdl.config.d_model
    print(f"Whisper encoder hidden dim: {D}")

    chunk_samples = WHISPER_CHUNK_SEC * SR

    @torch.no_grad()
    def whisper_embed(y):
        if len(y) <= chunk_samples:
            chunks = [y]
        else:
            chunks = [y[i:i+chunk_samples] for i in range(0, len(y), chunk_samples)]
        embs = []
        for c in chunks:
            if len(c) < int(SR * 0.5): continue
            feat = proc(c, sampling_rate=SR, return_tensors='pt').input_features.to(DEVICE)
            out  = mdl.encoder(feat)
            embs.append(out.last_hidden_state.mean(dim=1).squeeze(0).cpu().numpy())
        if not embs: return None
        return np.mean(np.stack(embs), axis=0)

    for m in need_wh:
        print(f"\nExtracting Whisper: {m['name']} ({len(m['files'])} files)...")
        rows = []
        for fp in tqdm(m['files'], desc=m['name']):
            y = load_audio_16k(fp)
            if y is None: continue
            e = whisper_embed(y)
            if e is None: continue
            row = {'filename': fp.name}
            for i, v in enumerate(e): row[f'whisper_{i}'] = round(float(v), 6)
            rows.append(row)
        pd.DataFrame(rows).to_csv(m['whisper'], index=False)
        print(f"  Saved: {m['whisper'].name} ({len(rows)} rows, {D} dims)")

    del mdl, proc
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    gc.collect()
else:
    print('All Whisper caches present.')

In [ ]:
X_tr, y_tr, X_te, y_te, _, te_df, _ = build_xy_from_cache('whisper_', folders, 'whisper')
print(f"Whisper:  train={X_tr.shape}  test={X_te.shape}")
res_wh = train_xgb_rf(X_tr, y_tr, X_te, y_te, tag='whisper')
save_results(res_wh, te_df, y_te)

## 5. Overall Comparison

In [ ]:
cmp_df = pd.DataFrame(all_results).sort_values('f1', ascending=False).reset_index(drop=True)
cmp_df.to_csv(SAVE_DIR / 'encoder_comparison.csv', index=False)

print('='*110)
print('  ENCODER COMPARISON  (sorted by F1)')
print('='*110)
base_cols = ['tag','n_feat','thr','f1','precision','recall','accuracy','tp','fp','fn','tn']
print(cmp_df[[c for c in base_cols if c in cmp_df.columns]].to_string(index=False))
print('='*110)

rp_cols = [f'rec@P{int(p*100)}' for p in PREC_TARGETS if f'rec@P{int(p*100)}' in cmp_df.columns]
if rp_cols:
    print('\n' + '='*110)
    print('  RECALL @ PRECISION TARGETS  (sorted by F1)')
    print('='*110)
    print(cmp_df[['tag','f1','precision','recall'] + rp_cols].to_string(index=False))
    print('='*110)

print(f"\nSaved -> {SAVE_DIR / 'encoder_comparison.csv'}")

summary = {
    'train_folders': TRAIN_FOLDERS,
    'test_folder':   TEST_FOLDER,
    'test_ratio':    TEST_RATIO if not TEST_FOLDER else None,
    'n_train': int(len(train_files)),
    'n_test':  int(len(test_files)),
    'models': {'wavlm': WAVLM_MODEL, 'wav2vec2': WAV2VEC2_MODEL, 'whisper': WHISPER_MODEL},
    'prec_targets': PREC_TARGETS,
    'results': cmp_df.to_dict(orient='records'),
}
with open(SAVE_DIR / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print(f"Saved -> {SAVE_DIR / 'summary.json'}")